In [1]:
import json
import pymupdf
import os
import re
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from docx import Document
from datetime import datetime

In [2]:
DATA_DIR = Path("data")
STATS_DIR = Path("stats")
ARTICLE_PATTERN = re.compile(r"\b(?:Article\s+\d+|Art\.\s+\d+)")

In [3]:
# get all metadata from last json file

most_recent_file = max(
    (f for f in STATS_DIR.iterdir() if f.suffix == ".json"),
    key=lambda f: f.stat().st_mtime
)

with open(most_recent_file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [4]:
# display stats on metadata

unique_refs = set()
for metadata_dict in data.values():
    for ref in metadata_dict.get("refs"):
        key = (
            ref.get("lex_id"),
            ref.get("lex_type"),
            ref.get("lex_number"),
            ref.get("lex_lang"),
        )
        unique_refs.add(key)
lang_counts = Counter(ref[3] for ref in unique_refs)
print(f"Number of source documents: {len(unique_refs)}")
print(f"Languages inside the corpus: {list(lang_counts.keys())} (size={len(lang_counts)})")
for lang, nb in lang_counts.items():
        print(f"  - {lang}: {nb}")

ref_occurences_counter = defaultdict(list)
lang_mismatch_occurences_counter = defaultdict(list)
for metadata_dict in data.values():
    refs = metadata_dict.get("refs")
    cat = metadata_dict.get("cat")
    if len(refs) > 1:
        ref_occurences_counter[cat].append(len(refs))
    langs = set()
    for ref in refs:
        langs.add(ref.get("lex_lang"))
    if len(langs) > 1:
        lang_mismatch_occurences_counter[cat].append(len(langs))
print(f"\nNumber of documents referenced more than once grouped by category (already handled):")
for cat, list_duplications in ref_occurences_counter.items():
    print(f"  - {cat}: {len(list_duplications)}")
print(f"Number of mismatch between languages inside the corpus grouped by category:")
for cat, list_lang_mismatch in lang_mismatch_occurences_counter.items():
    print(f"  - {cat}: {len(list_lang_mismatch)}")

corpus_size = len(data)
print(f"\nNumber of documents inside the corpus: {corpus_size}\n")

metadatas = [
    {"subject": "Categories", "key_in_dict": "cat"},
    {"subject": "Sources", "key_in_dict": "source"},
    {"subject": "Content formats", "key_in_dict": "content_format"},
]
for metadata in metadatas:
    counts = Counter(metadata_dict.get(metadata.get("key_in_dict")) for metadata_dict in data.values())
    print(f"{metadata.get("subject")} inside the corpus: {list(counts.keys())} (size={len(counts)})")
    for key, nb in counts.items():
        print(f"  - {key}: {nb}")

print(f"\nExample of an item in json file: {list(data.items())[42]}")

metatdata_df = pd.DataFrame(metadatas)
os.makedirs(STATS_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_stats_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata_stats.csv")
metatdata_df.to_csv(metadata_stats_filename, index=False)

Number of source documents: 338
Languages inside the corpus: ['en', 'fr'] (size=2)
  - en: 169
  - fr: 169

Number of documents referenced more than once grouped by category (already handled):
  - lex: 23
  - appendix: 13
  - summary: 2
Number of mismatch between languages inside the corpus grouped by category:
  - lex: 23
  - appendix: 10
  - summary: 2

Number of documents inside the corpus: 659

Categories inside the corpus: ['summary', 'lex', 'appendix'] (size=3)
  - summary: 336
  - lex: 276
  - appendix: 47
Sources inside the corpus: ['polylex', 'fedlex', 'others'] (size=3)
  - polylex: 336
  - fedlex: 37
  - others: 286
Content formats inside the corpus: ['txt', 'pdf', 'docx'] (size=3)
  - txt: 336
  - pdf: 321
  - docx: 2

Example of an item in json file: ('Regulations concerning the organisation of EPFL Core Facilities\nThe present regulations define the structure and organisation of the EPFL Core Facilities and govern the tasks and areas of authority of their bodies.', {'doc_

In [5]:
# save stats on metadata in csv file

metadatas_stats = {}
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

metadatas_stats["timestamp"] = timestamp

metadatas_stats["nb_unique_refs"] = len(unique_refs)

metadatas_stats["nb_languages"] = len(lang_counts)
for lang, nb in lang_counts.items():
    metadatas_stats[f"lang_{lang}"] = nb

metadatas_stats["corpus_size"] = len(data)

for cat, vals in ref_occurences_counter.items():
    metadatas_stats[f"multi_ref_{cat}"] = len(vals)

for cat, vals in lang_mismatch_occurences_counter.items():
    metadatas_stats[f"lang_mismatch_{cat}"] = len(vals)

for metadata in metadatas:
    key_name = metadata["key_in_dict"]
    counts = Counter(metadata_dict.get(key_name) for metadata_dict in data.values())
    metadatas_stats[f"{key_name}_nb_unique"] = len(counts)
    for key, nb in counts.items():
        metadatas_stats[f"{key_name}_{key}"] = nb

metatdata_df = pd.DataFrame([metadatas_stats])

os.makedirs(STATS_DIR, exist_ok=True)
metadata_stats_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata_stats.csv")
metatdata_df.to_csv(metadata_stats_filename, index=False)

In [6]:
# save stats on doc content in csv file

def count_ratio_alnum_chars(content):
    content_length = len(content) if len(content) > 0 else 1
    nb_alnum_chars = 0
    nb_al_chars = 0
    for char in content:
        if char.isalnum():
            nb_alnum_chars += 1
            if char.isalpha():
                nb_al_chars += 1
    ratio_alnum_chars = nb_alnum_chars / content_length
    ratio_al_chars = nb_al_chars / content_length
    ratio_num_chars = (nb_alnum_chars - nb_al_chars) / content_length
    return ratio_alnum_chars, ratio_al_chars, ratio_num_chars

def compute_stats(filename, nb_pages, no_page, content):
    nb_chars = len(content)
    ratio_alnum, ratio_al, ratio_num = count_ratio_alnum_chars(content)
    article_matches = ARTICLE_PATTERN.findall(content)
    return {
        "filename": filename,
        "nb_pages": nb_pages,
        "no_page": no_page,
        "nb_chars": nb_chars,
        "ratio_alnum": ratio_alnum,
        "ratio_al": ratio_al,
        "ratio_num": ratio_num,
        "nb_articles_match": len(article_matches)
    }

stats = []

for file in DATA_DIR.iterdir():
    if file.suffix == ".txt":
        nb_pages = 1
        no_page = 1
        content = file.read_text(encoding="utf-8")
        stats.append(compute_stats(file.name, nb_pages, no_page, content))
    elif file.suffix == ".docx":
        nb_pages = None
        no_page = 1
        doc = Document(file)
        content = "\n".join(p.text for p in doc.paragraphs)
        stats.append(compute_stats(file.name, nb_pages, no_page, content))
    elif file.suffix == ".pdf":
        with pymupdf.open(file) as f:
            nb_pages = f.page_count
            for no_page, page in enumerate(f):
                page_content = page.get_text()
                # TODO : use https://pymupdf.readthedocs.io/en/latest/page.html#Page.get_textpage_ocr and https://stackoverflow.com/questions/76834972/how-can-i-run-pytesseract-python-library-in-ubuntu-22-04
                # FIXME : pas de tres bons resultats avec cette methode...
                #if len(page_content) == 0:
                #    text_page = page.get_textpage_ocr(flags=0, language='fra', dpi=600, full=True)
                #    page_content = page.get_text(textpage=text_page)
                #    with open(os.path.join(file.stem + ".txt"), "a") as content:
                #        content.write(page_content)
                stats.append(compute_stats(file.name, nb_pages, no_page + 1, page_content))
    else:
        print(f"Error while reading {file}: format not supported")

df_stats = pd.DataFrame(stats)

os.makedirs(STATS_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
content_stats_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_content_stats.csv")
df_stats.to_csv(content_stats_filename, index=False)

In [7]:
# stats on volume by file

df_stats_by_file = (
    df_stats.groupby("filename", as_index=False).agg(
          nb_pages=pd.NamedAgg(column="nb_pages", aggfunc="first"),
          nb_chars=pd.NamedAgg(column="nb_chars", aggfunc="sum"),
          ratio_alnum=pd.NamedAgg(column="ratio_alnum", aggfunc="mean"),
          ratio_al=pd.NamedAgg(column="ratio_al", aggfunc="mean"),
          ratio_num=pd.NamedAgg(column="ratio_num", aggfunc="mean"),
          nb_articles_match=pd.NamedAgg(column="nb_articles_match", aggfunc="sum")
      )
)

nb_total_pages = int(df_stats_by_file["nb_pages"].sum())
print(f"There are {nb_total_pages} pages in total.")
nb_total_char = df_stats_by_file["nb_chars"].sum()
print(f"There are {nb_total_char} characters in total.")

# TODO : analyser distributions ci-dessous
df_stats_by_file

In [8]:
# stats on volume by page

df_stats_by_page = df_stats[
    ["filename", "no_page", "nb_chars", "ratio_alnum", "ratio_al", "ratio_num", "nb_articles_match"]
]

# TODO : analyser distributions ci-dessous
df_stats_by_page

There are 3195 pages in total.
There are 6959027 characters in total.


,filename,nb_pages,nb_chars,ratio_alnum,ratio_al,ratio_num,nb_articles_match
0,00d77a9657794a5b8cfbeb648cc3a9af.txt,1.0,522,0.833333,0.833333,0.000000,0
1,0118a9090dee3aa27b30d3f0fb62ef8f.pdf,3.0,9174,0.789202,0.767200,0.022001,8
2,0220a989c702fc75aa9554114aa01587.pdf,14.0,25556,0.748145,0.664031,0.084115,0
3,022f4eebd9eeb19623e57db1fb65a875.txt,1.0,790,0.826582,0.826582,0.000000,0
4,023a69ada015aea561f0ef66c0a35eec.txt,1.0,259,0.810811,0.779923,0.030888,0
...,...,...,...,...,...,...,...
653,fed8a1bfce8f2a3b7a261ea0110787a6.pdf,94.0,292655,0.791067,0.768844,0.022223,0
654,fef751c40a0702a8a0116e9f12f28a5b.txt,1.0,455,0.832967,0.832967,0.000000,0
655,fef7a5f1b04ef1a8f6273ed81db683c1.txt,1.0,439,0.804100,0.794989,0.009112,0
656,ff51f9376a472427cd101274e5469d68.pdf,15.0,33899,0.785522,0.758721,0.026802,42


In [9]:
# TODO : calculer longueur moyenne des mots, proportion de mots inconnus (via dictionnaire, pour savoir si noms propres ou OCR raté ou autres langues) et ratio mots uniques / total (trop élevé = bruit)
# TODO : cleaner les contenus (tableaux et autres)

,filename,no_page,nb_chars,ratio_alnum,ratio_al,ratio_num,nb_articles_match
0,580bbcfecee341f8a64b0e0f5e11a5c2.txt,1,290,0.831034,0.831034,0.000000,0
1,28ad68802c3d8a57681ed18d8b3528f3.txt,1,849,0.808009,0.782097,0.025913,0
2,32f5618496692a491d692ff60dd08b33.pdf,1,1644,0.776764,0.743917,0.032847,0
3,32f5618496692a491d692ff60dd08b33.pdf,2,2352,0.782738,0.757228,0.025510,0
4,32f5618496692a491d692ff60dd08b33.pdf,3,1858,0.785253,0.760495,0.024758,0
...,...,...,...,...,...,...,...
3192,ab8bde56e0d887219dddbf9a8a0b0496.pdf,4,2656,0.783133,0.758660,0.024473,6
3193,ab8bde56e0d887219dddbf9a8a0b0496.pdf,5,3229,0.795602,0.778569,0.017033,4
3194,ab8bde56e0d887219dddbf9a8a0b0496.pdf,6,2139,0.783544,0.751286,0.032258,3
3195,ee9081e72dd10257b4fc7fdae665beed.txt,1,704,0.815341,0.795455,0.019886,0


In [10]:
# FIXME : base sur https://medium.com/@alice.yang_10652/extract-text-from-images-and-scanned-pdfs-with-python-2087cb1e0a7b

# from spire.pdf import *
# from spire.ocr import *
#
# SCANNED_DATA_DIR = Path('text_from_scanned_pdfs')
# os.makedirs(SCANNED_DATA_DIR, exist_ok=True)
#
# scanned_files = df_stats_by_file['filename'][df_stats_by_file['content_length'] == 0].values
#
# def convert_page_to_image(file, page_index):
#     return file.SaveAsImage(page_index)
#
# def recognize_text_from_image(image_name, language, model_path):
#     scanner = OcrScanner()
#     configure_options = ConfigureOptions()
#     configure_options.Language = language
#     configure_options.ModelPath = model_path
#     scanner.ConfigureDependencies(configure_options)
#
#     scanner.Scan(str(image_name))
#     data = scanner.Text.ToString()
#     return data
#
# for file in scanned_files:
#     filepath = os.path.join(DATA_DIR, file)
#     pdf = PdfDocument()
#     pdf.LoadFromFile(str(filepath))
#
#     filename = os.path.splitext(os.path.basename(filepath))[0]
#     text_filename = filename + ".txt"
#
#     image_dir = os.path.join(SCANNED_DATA_DIR, filename)
#     os.makedirs(image_dir, exist_ok=True)
#
#     with open(os.path.join(SCANNED_DATA_DIR, text_filename), 'a', encoding='utf-8') as writer:
#         for page_index in range(pdf.Pages.Count):
#             image = convert_page_to_image(pdf, page_index)
#             image_name = os.path.join(image_dir, f"{page_index}.png")
#             image.Save(image_name)
#             recognized_text = recognize_text_from_image(image_name, 'French', r'ocr_model')
#             #writer.write(f'Page {page_index + 1}:\n')
#             writer.write(recognized_text)
#             #writer.write('\n\n')